In [8]:
import pandas as pd
import random
from faker import Faker
fake = Faker()
social_network_strength = 2.5
number_of_people = 300

In [9]:
#generation of random people and their pronouns,
pronoun_groups = [
    "she/her",
    "he/him",
    "they/them",
    "xe/xer",
    "she/they",
    "he/they"
    
]

people = pd.DataFrame({
    "ID":range(1, number_of_people + 1),
    "Name": [fake.name() for _ in range(number_of_people)],
    "Pronouns": [random.choice(pronoun_groups) for _ in range(number_of_people)]
})

In [16]:
# Copy user group for 
def map_user_group(pronoun_str):
    pronoun_str = pronoun_str.lower()
    if pronoun_str in ["he/him", "he"]:
        return "he"
    elif pronoun_str in ["she/her", "she"]:
        return "she"
    elif pronoun_str in ["they/them", "they"]:
        return "they"
    elif pronoun_str in ["xe/xer", "xe"]:
        return "xe"
    elif pronoun_str in ["she/they"]:
        return "she/they"
    elif pronoun_str in ["he/they"]:
        return "he/they"
    else:
        return pronoun_str  # keep unknown as is

people["User_Group"] = people["Pronouns"].apply(map_user_group)

# Each row will have a list of base pronouns
people["Base_Pronoun_List"] = people["User_Group"].apply(lambda x: x.split("/"))

#determines social network seperation
def pronoun_similarity(p1, p2):
    set1 = set(p1.split("/"))
    set2 = set(p2.split("/"))
    shared = len(set1.intersection(set2))
    
    if shared > 0:
        return social_network_strength
    else:
        return 1.0

In [17]:
#extending generation process to have probabilitic parameters
#random connections now, extend the probabiltiies of pronouns

In [19]:
# Homophily-based network generation (affinity for similar pronouns)
connective_lst = []

for person_id in people["ID"]:
    
    number_of_connections = random.randint(3, 6)  # increased size
    
    person_pronoun = people.loc[
        people["ID"] == person_id, "User_Group"
    ].values[0]
    
    potential_people = people[people["ID"] != person_id]
    
    weights = []
    for _, row in potential_people.iterrows():
        similarity_weight = pronoun_similarity(
            person_pronoun,
            row["User_Group"]
        )
        weights.append(similarity_weight)
    
    total_weight = sum(weights)
    probabilities = [w / total_weight for w in weights]
    
    connect_ids = random.choices(
        population=list(potential_people["ID"]),
        weights=probabilities,
        k=number_of_connections
    )
    
    for conid in connect_ids:
        connective_lst.append({
            "FromID": person_id,
            "ToID": conid
        })

connections = pd.DataFrame(connective_lst)

In [20]:
#merging both people and pronouns into a singular excel sheet
merging = connections.merge(
    people, left_on="FromID", right_on="ID"
).merge(
    people, left_on="ToID", right_on="ID", suffixes=("_From", "_To")
)

final = merging[[
    "Name_From", "Pronouns_From",
    "Name_To", "Pronouns_To"
]]
final.head()

,Name_From,Pronouns_From,Name_To,Pronouns_To
0,Jaime Ortiz,they/them,Amy Oliver,they/them
1,Jaime Ortiz,they/them,Gary Griffin,she/they
2,Jaime Ortiz,they/them,Wayne Thompson,she/her
3,Jaime Ortiz,they/them,Timothy Bowen,she/they
4,Dr. Michael Lopez,xe/xer,Wayne Thompson,she/her


In [21]:

# Explode so each base pronoun is a separate row
exploded = people.explode("Base_Pronoun_List")

pronoun_matrix = pd.crosstab(
    exploded["User_Group"],
    exploded["Base_Pronoun_List"],
    normalize="index"
)

# Ensure consistent column order
pronoun_matrix = pronoun_matrix.reindex(
    columns=["he", "she", "they", "xe"], fill_value=0
)

# Add user counts
group_counts = people["User_Group"].value_counts()
pronoun_matrix["User_Count"] = group_counts

# Move User_Count to first column
cols = ["User_Count"] + [col for col in pronoun_matrix.columns if col != "User_Count"]
pronoun_matrix = pronoun_matrix[cols]


In [22]:
pronoun_counts = people["Pronouns"].value_counts()
pronoun_prob = people["Pronouns"].value_counts(normalize=True)
pronoun_percent = pronoun_prob * 100

#connection pronoun probabilities
from_prob = final["Pronouns_From"].value_counts(normalize=True)
to_prob = final["Pronouns_To"].value_counts(normalize=True)

In [23]:
#excel export
final.to_csv("people_connections_social_network.csv", index=False, encoding='utf-8-sig')
pronoun_matrix.to_csv("pronoun_probability_constraints.csv")

